In [4]:
%run ../shared_notebook_setup.py

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=DATABRICKS_MODEL,
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# StrOutputParser converts LLM output into a plain string by extracting the text content.
parser = StrOutputParser()

## Example 1: Basic PromptTemplate
Use a single placeholder to generate a reusable instruction prompt.

In [ ]:
template1 = PromptTemplate.from_template(
    "Explain {topic} in exactly 2 concise bullet points."
)

# Below is to demonstrate what the prompt looks like after formatting with a specific topic. 
# Useful for debugging and understanding the prompt structure.

# PromptTemplate.format() method to create a prompt for a specific input
formatted1 = template1.format(topic="prompt engineering")
print("Formatted prompt:\n")
print(formatted1)

chain1 = template1 | llm | parser
print("\nModel output:\n")
print(chain1.invoke({"topic": "prompt engineering"}))

Formatted prompt:

Explain prompt engineering in exactly 2 concise bullet points.

Model output:

* Prompt engineering is the strategic design of text inputs to guide AI models toward accurate, relevant outputs.
* It optimizes performance by refining context, instructions, and constraints through iterative testing.


## Example 2: Multiple Input Variables

In [ ]:
template2 = PromptTemplate(
    input_variables=["audience", "topic", "tone"],
    template="Write a {tone} 4-line explanation of {topic} for {audience}.",
)

print(template2.format(
    audience="new Python developers",
    topic="LangChain prompt templates",
    tone="friendly",
))

chain2 = template2 | llm | parser
print("\nModel output:\n")
print(chain2.invoke({
    "audience": "new Python developers",
    "topic": "LangChain prompt templates",
    "tone": "friendly",
}))

## Example 3: ChatPromptTemplate With Roles

In [8]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a concise teaching assistant."),
    ("human", "Give 3 tips for learning {topic}."),
])

messages = chat_template.format_messages(topic="LangChain")
print("Preview messages:")
for m in messages:
    print(f"- {m.type}: {m.content}")

chain3 = chat_template | llm | parser
print("\nModel output:\n")
print(chain3.invoke({"topic": "LangChain"}))

Preview messages:
- system: You are a concise teaching assistant.
- human: Give 3 tips for learning LangChain.

Model output:

Here are 3 concise tips for learning LangChain effectively:

1.  **Master the Primitives First**
    Before diving into complex Agents, understand core components: **Models**, **Prompts**, and **Outputs**. Build simple chains that just pass a string to an LLM and back.

2.  **Learn LCEL Early**
    Focus on **LangChain Expression Language (LCEL)**. It is the modern, declarative way to build chains (using `|` syntax) and is more stable and testable than older imperative methods.

3.  **Build Incrementally**
    Start with a script that answers one question using a standard model. Then, add one feature at a time (e.g., memory, external data via RAG) rather than trying to build a full application immediately.


## Example 4: Partial Prompt Templates
Use `.partial(...)` to lock in common values and only provide changing inputs later.

In [ ]:
base_template = PromptTemplate.from_template(
    "You are a {role}. Explain {concept} for a {audience} in 3 short bullet points."
)

# partial() method to create a new template with role variables fixed
# this cannot be modified when invoking the chain, so it enforces a specific role for the LLM output.
teacher_template = base_template.partial(role="helpful instructor")

print(teacher_template.format(
    concept="temperature in language models",
    audience="beginner",
))

chain4 = teacher_template | llm | parser
print("\nModel output:\n")
print(chain4.invoke({
    "concept": "temperature in language models",
    "audience": "beginner",
}))

## Example 5: Batch Invocation
Use `.batch(...)` when you want to run one template over many inputs.

In [ ]:
batch_inputs = [
    {"topic": "prompt templates"},
    {"topic": "chat prompts"},
    {"topic": "LCEL"},
]

template5 = PromptTemplate.from_template(
    "In one sentence, explain {topic} in simple language."
)
chain5 = template5 | llm | parser

outputs = chain5.batch(batch_inputs)
for i, out in enumerate(outputs, start=1):
    print(f"{i}. {out}")